<a href="https://colab.research.google.com/github/farithjhg/aws-bedrock-agentcore-course/blob/main/v7_aprender_haciendo/labs/06_contexto_y_coste.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

🎬 [La escena en la presentación](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/presentacion_nova_assist.html#cap06) · 🧭 [Teoría del capítulo](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/cap06-contexto-y-coste.html) · 🗺️ [El mapa](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/index.html) · ⬅️ [Cap. 5](https://colab.research.google.com/github/farithjhg/aws-bedrock-agentcore-course/blob/main/v7_aprender_haciendo/labs/05_memoria_largo_plazo.ipynb) · ➡️ [Cap. 7](https://colab.research.google.com/github/farithjhg/aws-bedrock-agentcore-course/blob/main/v7_aprender_haciendo/labs/07_gateway.ipynb)

# Capítulo 6 — La cuenta del taxímetro

**Formación Amazon Bedrock AgentCore · octubre 2026 · Caso de uso: Nova Assist (TiendaNova)**

> 🎬 **Escena.** Finanzas pregunta por qué la factura de Bedrock se ha triplicado en un mes. Nadie ha añadido clientes; hemos añadido memoria, herramientas e historial.

| | |
|---|---|
| **🧠 Componente agéntico** | Contexto: qué ve el modelo en cada llamada |
| **La necesidad** | Controlar qué entra en la ventana de contexto: menos coste, menos latencia y respuestas más centradas |
| **Lo que construimos** | *Prompt caching* real (prefijo estable + puntos de caché), contexto dinámico separado y ventana deslizante del historial |
| **🏠 En local** | ✅ Funciona igual en local y en la nube: es diseño del prompt y del historial. |
| **☁️ Por qué AWS al salir a Internet** | La API del modelo no guarda estado: **cada llamada reenvía todo**. En una conversación de 20 turnos se paga muchas veces el mismo contenido. En producción se mide (*tokens* por conversación), se cachea el prefijo y se limita el historial. |
| **En el ordenador sería…** | La **gestión de memoria del sistema operativo**: decidir qué se carga en la RAM, en qué orden y qué se descarga |
| **Requisitos del caso** | Eficiencia y coste (D6) |
| **Prerrequisito · duración** | Capítulo 5 · 45 min |
| **⚠️ Dónde se rompe la analogía** | Un sistema operativo pagina la memoria sin perder datos; **recortar el contexto sí pierde información**. Hay que elegir qué se olvida. |

### ☁️ ¿Estás en Google Colab? Ejecuta primero esta celda

En tu equipo no hace nada. En Colab descarga el curso, monta tu **Google Drive** (ahí se guarda tu estado entre capítulos: `MyDrive/NovaAssist_curso`), instala las dependencias y la AgentCore CLI 0.31.1 y carga tus credenciales de AWS desde los **Secretos** de Colab. Tarda 2–3 minutos, porque cada notebook abre una máquina nueva: mientras tanto, lee la escena.

In [ ]:
# ☁️ Solo en Google Colab (en tu equipo no hace nada)
import os, sys, subprocess
if "google.colab" in sys.modules:
    CURSO = "/content/aws-bedrock-agentcore-course"
    if not os.path.isdir(CURSO):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", "main",
                        "https://github.com/farithjhg/aws-bedrock-agentcore-course.git", CURSO], check=True)
    os.chdir(f"{CURSO}/v7_aprender_haciendo/labs")
    sys.path.insert(0, os.getcwd())
    import colab_nova
    colab_nova.preparar(capitulo=6)

In [ ]:
# Configuración común: ejecuta siempre esta celda primero
import os
from lab_utils import *

os.environ.setdefault("AWS_REGION", region())
cuaderno()

> 🎮 **¿Llegas tarde o algo se rompió?** La celda siguiente comprueba que tienes todo lo necesario. Si no, descomenta `ponerse_al_dia(6)`: carga la partida guardada al final del capítulo anterior o, si no existe, escribe el código de referencia y te dice qué recursos faltan y en qué capítulo se crean.

In [ ]:
requisitos(6)
# ponerse_al_dia(6)   # ← descomenta si llegas tarde o algo se rompió

## 1. El histórico del taxímetro

In [ ]:
taximetro_historico()

## 2. Experimento: una conversación larga

Ocho turnos en la **misma** sesión. Fíjate en cómo crecen los *tokens* de entrada **por turno**: cada mensaje reenvía el historial completo.

> ✋ **Predice antes de ejecutar.** Antes de mirar: ¿cuánto costará el turno 8 frente al turno 1? ¿Lo mismo? ¿El doble? ¿Cinco veces más? Apunta tu apuesta; la compararemos.

In [ ]:
CONVERSACION = ["Hola, soy Laura", "¿Dónde está mi pedido TN-1001?", "¿Y quién lo lleva?",
                "¿Qué artículos tiene el TN-1003?", "Quiero devolver la cafetera", "¿Cuánto tardará el reembolso?",
                "¿Me avisáis por WhatsApp?", "Por cierto, ¿cómo me llamo?"]
sid_antes = new_session_id()
for mensaje in CONVERSACION:
    invocar(mensaje, session_id=sid_antes)

In [ ]:
por_turno = turnos(sid_antes)
print(f"Turno 1: {por_turno[0]['entrada']:,} tokens de entrada · turno 8: {por_turno[-1]['entrada']:,} "
      f"→ {por_turno[-1]['entrada'] / max(por_turno[0]['entrada'], 1):.1f} veces más")

Compara la cifra con tu apuesta. Los tokens de entrada **por turno** crecen con cada mensaje. Nadie ha añadido clientes: hemos añadido historial, y lo reenviamos entero cada vez.

> 🧭 **¿Por qué ha pasado?** Abre la teoría: [**La gestión de memoria del agente**](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/cap06-contexto-y-coste.html). Mueve los mandos del contexto (ventana, resultados compactos, memoria, caché) y cambia a la vista de Finanzas. Vuelve cuando sepas responder a esta pregunta: **¿qué parte del contexto se reenvía en cada llamada sin cambiar, y cuál podrías dejar de enviar sin que el agente pierda lo importante?**

**En dos líneas:** la API del modelo no guarda estado, así que **cada llamada reenvía todo** (prompt, herramientas, memoria, historial) y hay varias llamadas por turno: el coste total crece más o menos con el cuadrado de la conversación. Ingeniería de contexto es decidir **qué** entra, **en qué orden** y **con qué límites**: prefijo estable primero (cacheable), contenido dinámico después, historial con ventana o resumen, resultados compactos y `top_k` pequeño. Más contexto no es mejor: lo irrelevante diluye la atención.

## 3. Prefijo estable, puntos de caché y ventana deslizante

Tres cambios, cada uno en su módulo:

- **`prompts.py`**: solo lo estable (identidad, reglas, formato).
- **`contexto.py`**: el *system prompt* se envía como **bloques**: primero el prefijo estable, después un **punto de caché** y al final lo que cambia en cada sesión (fecha, canal, idioma). También fija la **ventana**: 12 mensajes de historial.
- **`modelo.py`**: el modelo con *prompt caching* (`CacheConfig(strategy="auto")`): Strands pone puntos de caché tras las herramientas, tras el prefijo y al final de la conversación. Lo ya enviado se relee de la caché a una fracción del precio (×0,1) en lugar de procesarse de nuevo; escribir en la caché cuesta algo más (×1,25), una sola vez cada 5 minutos.

In [ ]:
%%writefile NovaAssist/app/NovaAgent/prompts.py
# Prefijo ESTABLE (cacheable). El contexto dinámico se añade después, en main.py.
SYSTEM_PROMPT = """<identidad>
Eres Nova Assist, el asistente de postventa de TiendaNova. En tu primer mensaje te presentas como asistente de IA.
</identidad>

<reglas>
1. Nunca inventes estados, fechas ni importes: usa siempre las herramientas.
2. Los importes se expresan en euros al cliente, pero las herramientas trabajan en céntimos.
3. Para dudas de política (devoluciones, garantías, envíos) consulta la base de conocimiento y cita el documento.
4. Si una herramienta es denegada por política o el caso supera tus límites, usa escalar_a_humano con un resumen claro.
5. No pidas ni repitas datos de tarjetas de pago.
</reglas>

<formato>
Respuestas breves (máximo 5 frases), con el siguiente paso claro para el cliente.
</formato>"""

In [ ]:
%%writefile NovaAssist/app/NovaAgent/contexto.py
"""Contexto dinámico por invocación y límite del historial (capítulo 6)."""
from datetime import date

from prompts import SYSTEM_PROMPT

# Mensajes del historial que se reenvían en cada llamada. Un turno con herramientas suma 4 mensajes
# (pregunta, petición de herramienta, resultado y respuesta), así que 12 mensajes ≈ los últimos 3-6 turnos.
VENTANA_MENSAJES = 12


def contexto_sesion(canal: str, idioma: str) -> str:
    return (
        "<contexto_sesion>\n"
        f"fecha_actual: {date.today().isoformat()}\n"
        f"canal: {canal}\n"
        f"idioma_preferido: {idioma}\n"
        "</contexto_sesion>"
    )


def system_prompt(payload: dict) -> list[dict]:
    """Prefijo ESTABLE primero (cacheable), después un punto de caché y lo que cambia en cada sesión."""
    return [
        {"text": SYSTEM_PROMPT},
        {"cachePoint": {"type": "default"}},
        {"text": contexto_sesion(payload.get("canal", "web"), payload.get("idioma", "es"))},
    ]

In [ ]:
%%writefile NovaAssist/app/NovaAgent/modelo.py
"""El modelo del agente: inference profile, reintentos adaptativos y prompt caching (capítulo 6)."""
from botocore.config import Config
from strands.models import BedrockModel, CacheConfig

from settings import MODEL_ID, REGION

REINTENTOS = Config(retries={"mode": "adaptive", "max_attempts": 8})


def crear_modelo(temperatura: float = 0.2) -> BedrockModel:
    # strategy="auto": puntos de caché tras las herramientas, tras el prefijo estable y al final de la conversación,
    # para que cada llamada relea de la caché lo que ya se envió en la anterior.
    return BedrockModel(model_id=MODEL_ID, region_name=REGION, temperature=temperatura,
                        boto_client_config=REINTENTOS,
                        cache_config=CacheConfig(strategy="auto", tools_ttl=True))

In [ ]:
%%writefile NovaAssist/app/NovaAgent/main.py
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands import Agent
from strands.agent.conversation_manager import SlidingWindowConversationManager

from contexto import VENTANA_MENSAJES, system_prompt
from herramientas import HERRAMIENTAS
from memoria import crear_session_manager, retrieval_config
from modelo import crear_modelo
from respuesta import respuesta

app = BedrockAgentCoreApp()


@app.entrypoint
def invoke(payload, context):
    actor_id = payload.get("cliente_id", "anonimo")  # ⚠️ provisional: en el capítulo 9 saldrá del token
    with crear_session_manager(actor_id, context.session_id, retrieval_config()) as sm:
        agent = Agent(
            model=crear_modelo(),
            system_prompt=system_prompt(payload),
            tools=HERRAMIENTAS,
            session_manager=sm,
            conversation_manager=SlidingWindowConversationManager(window_size=VENTANA_MENSAJES),
            callback_handler=None,
        )
        result = agent(payload.get("prompt", ""))
    return respuesta(result)


if __name__ == "__main__":
    app.run()

In [ ]:
cli('agentcore deploy --yes')

## 4. La misma conversación, con caché y ventana

> ✋ **Predice antes de ejecutar.** Con caché y una ventana de 12 mensajes: ¿cuánto costará ahora el turno 8 frente al turno 1? ¿Y recordará el nombre en el último turno, si el «Hola, soy Laura» del turno 1 ya ha salido de la ventana?

In [ ]:
sid_despues = new_session_id()
for mensaje in CONVERSACION:
    invocar(mensaje, session_id=sid_despues)
comparar_conversaciones(sid_antes, sid_despues, ("sin caché", "con caché"))
taximetro()

**Cómo leer la tabla:** la columna *entrada* son los *tokens* procesados a precio completo; *caché* son los releídos a ×0,1. Lo esperable:

- **La caché** aparece desde el segundo turno (o desde la segunda llamada de un turno con herramientas): lo ya enviado deja de pagarse entero. Si la columna de caché se queda a 0, el prefijo no llega al mínimo cacheable del modelo (unos 1.024 *tokens* en la mayoría de modelos Claude): la caché solo compensa con prefijos largos.
- **La ventana** frena el crecimiento a partir del turno en que el historial supera los 12 mensajes: se descartan los más antiguos.
- **El nombre:** el «Hola, soy Laura» ya no está en la ventana. Si aun así responde «Laura», no lo ha sacado del historial: se lo ha dado la **memoria a largo plazo** (capítulo 5), que lo extrajo en conversaciones anteriores. Recortar el contexto pierde información; lo importante tiene que vivir en otra parte.

> La ventana y la caché tiran en direcciones opuestas: cuando la ventana recorta, cambia el principio de la conversación y la caché de los mensajes deja de servir (la de herramientas y prompt sigue sirviendo). Por eso se mide.

**Más palancas:**
- **Resultados de herramienta compactos:** solo los campos necesarios.
- **`top_k` pequeño** en memoria y RAG.
- **Búsqueda semántica de herramientas** en el Gateway, en lugar de cargar todas (capítulo 7).

**Regla de diseño:** el plazo de devolución no se calcula en el prompt: lo calculará el código de la herramienta (capítulo 7). El modelo decide *qué* hacer; el código decide *si es válido*.

## Para pensar

- ¿Qué información sobra hoy en tu contexto?
- ¿Por qué el contenido dinámico va después del prefijo estable?

## 🔎 Lo que acabamos de descubrir

Hemos puesto orden en la RAM. Pero TiendaNova lanza una **app para repartidores** que necesita consultar envíos, y las herramientas están **pegadas al código de Nova**, con los datos dentro. ¿Copiamos todo en otro agente?

In [ ]:
deuda("D6", "parcial", 6, "caché y ventana reducen el coste por turno; aún no hay un límite por conversación (cap. 12)")
cuaderno()
taximetro_cerrar(6)
guardar_partida(6)

📚 **Para repasar más adelante:** la teoría de este capítulo está en [`teoria/cap06-contexto-y-coste.html`](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/cap06-contexto-y-coste.html) y el mapa completo en [`teoria/index.html`](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/index.html). La referencia por componente es la Parte I del documento.

---
➡️ **Siguiente capítulo:** `07_gateway.ipynb` — *Herramientas compartidas y gobernadas*

---
**Siguiente:** ➡️ [Capítulo 7 · Herramientas compartidas y gobernadas](https://colab.research.google.com/github/farithjhg/aws-bedrock-agentcore-course/blob/main/v7_aprender_haciendo/labs/07_gateway.ipynb) · 🎬 [su escena en la presentación](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/presentacion_nova_assist.html#cap07). En Colab, tu estado ya está en Drive: el capítulo siguiente lo recupera solo.